In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Load raw data
df = pd.read_csv('../data/cs-training.csv', index_col=0)
print(f"Original shape: {df.shape}")

# ============================================================
# STEP 1 - Remove invalid rows
# ============================================================

# Remove age = 0 (invalid)
df = df[df['age'] > 0]
print(f"After removing age=0: {df.shape}")

# ============================================================
# STEP 2 - Cap outliers
# ============================================================

# RevolvingUtilization - cap at 1.0 (it's a ratio, max should be 1)
df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(0, 1)

# Late payment features - cap at 10 (98 is data entry error)
late_cols = ['NumberOfTime30-59DaysPastDueNotWorse', 
             'NumberOfTimes90DaysLate', 
             'NumberOfTime60-89DaysPastDueNotWorse']
for col in late_cols:
    df[col] = df[col].clip(0, 10)

# DebtRatio - cap at 99th percentile
debt_cap = df['DebtRatio'].quantile(0.99)
df['DebtRatio'] = df['DebtRatio'].clip(0, debt_cap)

# MonthlyIncome - cap at 99th percentile
income_cap = df['MonthlyIncome'].quantile(0.99)
df['MonthlyIncome'] = df['MonthlyIncome'].clip(0, income_cap)

print(f"After capping outliers: {df.shape}")

# ============================================================
# STEP 3 - Impute missing values
# ============================================================

# NumberOfDependents - impute with median (0)
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

# MonthlyIncome - impute with median grouped by age bracket
df['age_bracket'] = pd.cut(df['age'], bins=[0, 30, 40, 50, 60, 70, 120], 
                            labels=['<30', '30-40', '40-50', '50-60', '60-70', '70+'])
df['MonthlyIncome'] = df.groupby('age_bracket')['MonthlyIncome'].transform(
    lambda x: x.fillna(x.median())
)
# Fill any remaining nulls with overall median
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df = df.drop('age_bracket', axis=1)

print(f"Missing values after imputation: {df.isnull().sum().sum()}")

# ============================================================
# STEP 4 - Feature engineering
# ============================================================

# Total late payments across all severity levels
df['total_late_payments'] = (df['NumberOfTime30-59DaysPastDueNotWorse'] + 
                              df['NumberOfTimes90DaysLate'] + 
                              df['NumberOfTime60-89DaysPastDueNotWorse'])

# Debt to income ratio
df['debt_to_income'] = df['DebtRatio'] / (df['MonthlyIncome'] + 1)

print(f"\nFinal shape: {df.shape}")
print(f"Features: {df.columns.tolist()}")

Original shape: (150000, 11)
After removing age=0: (149999, 11)
After capping outliers: (149999, 11)
Missing values after imputation: 0

Final shape: (149999, 13)
Features: ['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'total_late_payments', 'debt_to_income']


In [2]:
from sklearn.model_selection import train_test_split

# ============================================================
# STEP 5 - Split features and target
# ============================================================

feature_cols = ['RevolvingUtilizationOfUnsecuredLines', 'age', 
                'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 
                'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans',
                'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 
                'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents',
                'total_late_payments', 'debt_to_income']

X = df[feature_cols]
y = df['SeriousDlqin2yrs']

# Train/validation/test split - 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, 
                                                      random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, 
                                                  random_state=42, stratify=y_temp)

print(f"Training set: {X_train.shape} | Default rate: {y_train.mean():.2%}")
print(f"Validation set: {X_val.shape} | Default rate: {y_val.mean():.2%}")
print(f"Test set: {X_test.shape} | Default rate: {y_test.mean():.2%}")

# Save processed data
import os
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("\nProcessed data saved to data/processed/")

Training set: (104999, 12) | Default rate: 6.68%
Validation set: (22500, 12) | Default rate: 6.68%
Test set: (22500, 12) | Default rate: 6.68%

Processed data saved to data/processed/
